# Early-Exit TunedLens Speculative Decoding — Full Pipeline Runner

This notebook runs the modular `.py` implementation inside the `early_exit/` folder.

It walks through:

1. Environment setup
2. Importing the package
3. Loading model, tokenizer, and TunedLens
4. Loading text data
5. Running one smoke-test example
6. Running the full benchmark loop
7. Summarizing alpha, timing, and improvement
8. Optionally running the same pipeline through the CLI

The implementation is based on the original notebook idea:

> Use an early transformer layer + TunedLens as a draft model, then verify draft tokens with the full model using speculative decoding.

## 0. Folder expectation

Expected structure:

```text
your_project/
  early_exit/
    __init__.py
    config.py
    data.py
    modeling.py
    draft.py
    spec_decode.py
    benchmark.py
    run_benchmark.py
  early_exit_full_pipeline.ipynb
```

In Colab or a notebook environment, upload/unzip the `early_exit` folder next to this notebook.

In [ ]:
from pathlib import Path
import os
import sys
import json
import time

PROJECT_ROOT = Path.cwd()

# If you are running this notebook from /mnt/data or after upload,
# this tries to make sure the current folder is importable.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", PROJECT_ROOT)
print("Python path contains cwd:", str(PROJECT_ROOT) in sys.path)

early_exit_dir = PROJECT_ROOT / "early_exit"
print("early_exit folder exists:", early_exit_dir.exists())

## 1. Install dependencies

Run this cell once if the environment does not already have the dependencies.

For GPU runtime, make sure PyTorch is installed with CUDA support.

In [ ]:
# Uncomment if needed.
# !pip install -q transformers datasets tuned-lens accelerate numpy

## 2. Check GPU and package imports

This verifies that CUDA is available and imports the modular pipeline.

In [ ]:
import torch
import numpy as np
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA not available. Use a smaller model or set device='cpu'.")

In [ ]:
from early_exit.config import EarlyExitConfig
from early_exit.data import load_streaming_text_dataset
from early_exit.modeling import load_model_tokenizer_and_lens, get_transformer_layers
from early_exit.draft import generate_with_tuned_lens
from early_exit.spec_decode import speculative_decode_once, speculative_generate, DecodeStats
from early_exit.benchmark import benchmark_one_example, run_benchmark

print("early_exit imports: OK")

## 3. Hugging Face token

Do **not** hardcode your Hugging Face token in the notebook.

Recommended:

```bash
export HF_TOKEN="hf_..."
```

Or in Colab:

```python
import os
os.environ["HF_TOKEN"] = "hf_..."
```

Only set it below temporarily if you are in a private runtime.

In [ ]:
# Optional. Prefer environment variables/secrets.
# os.environ["HF_TOKEN"] = "hf_..."

print("HF_TOKEN exists:", bool(os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")))

## 4. Configure the experiment

Important parameters:

- `model_name`: base LM used both as full model and truncated early model
- `layer_index`: early layer used for TunedLens drafting
- `gamma`: number of draft tokens per speculative step
- `seqlen`: number of new tokens to generate per example
- `num_prefix_tokens`: prompt length
- `num_examples`: number of dataset examples to benchmark

For first run, keep this small.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

config = EarlyExitConfig(
    model_name="EleutherAI/pythia-1.4b-deduped",
    dataset_name="HuggingFaceFW/fineweb",
    dataset_config="CC-MAIN-2014-10",
    dataset_split="train",
    device=device,
    dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    gamma=4,
    seqlen=10,
    layer_index=3,
    num_prefix_tokens=10,
    num_examples=10,   # increase to 100 after smoke test
    seed=42,
)

config

## 5. Load model, tokenizer, and TunedLens

This may take time and GPU memory.

For Pythia-1.4B, the model layers are under:

```python
model.gpt_neox.layers
```

The helper `get_transformer_layers()` abstracts that path.

In [ ]:
transformers.set_seed(config.seed)

model, tokenizer, tuned_lens = load_model_tokenizer_and_lens(
    model_name=config.model_name,
    device=config.torch_device,
    dtype=config.dtype,
    hf_token=config.hf_token,
)

layers = get_transformer_layers(model)

print("model loaded:", config.model_name)
print("num layers:", len(layers))
print("expected layers:", model.config.num_hidden_layers)
print("vocab size:", model.config.vocab_size)
print("tokenizer pad_token_id:", tokenizer.pad_token_id)
print("tokenizer eos_token_id:", tokenizer.eos_token_id)

## 6. Load a streaming text dataset

The benchmark uses a streaming dataset so we do not download everything at once.

In [ ]:
ds = load_streaming_text_dataset(
    dataset_name=config.dataset_name,
    dataset_config=config.dataset_config,
    split=config.dataset_split,
)

example = next(iter(ds))
print(example.keys())
print(example["text"][:500])

## 7. Tokenize one example

We use only the first `num_prefix_tokens` tokens as the prefix.

In [ ]:
input_text = example["text"]

encoded = tokenizer(input_text, return_tensors="pt")
input_ids = encoded["input_ids"].to(config.torch_device)[0][: config.num_prefix_tokens]

print("input_ids shape:", tuple(input_ids.shape))
print("prefix tokens:", input_ids.tolist())
print("prefix text:")
print(tokenizer.decode(input_ids))

## 8. Draft tokens from an early layer with TunedLens

This calls:

```python
generate_with_tuned_lens(model, tuned_lens, layer_index, prefix_tokens, gamma)
```

Dimension flow:

```text
prefix_tokens:        [T]
model input:          [1, T]
hidden state:         [1, T, hidden_dim]
TunedLens logits:     [1, T, vocab]
next-token logits:    [1, vocab]
sampled draft token:  [1]
```

The function temporarily truncates the model to layers `0..layer_index`, generates `gamma` draft tokens, then restores the model layers.

In [ ]:
draft_out = generate_with_tuned_lens(
    model=model,
    tuned_lens=tuned_lens,
    layer_index=config.layer_index,
    prefix_tokens=input_ids,
    gamma=config.gamma,
)

draft_seq = draft_out["sequences"]
draft_scores = draft_out["scores"]

print("draft sequence shape:", tuple(draft_seq.shape))
print("number of score tensors:", len(draft_scores))
print("one score tensor shape:", tuple(draft_scores[0].shape))
print("drafted tokens:", draft_seq[0, -config.gamma:].tolist())
print("drafted text:")
print(tokenizer.decode(draft_seq[0]))
print("layers restored:", len(get_transformer_layers(model)) == model.config.num_hidden_layers)

## 9. Run one speculative decoding step

One speculative step does:

1. Draft `gamma` tokens from early layer + TunedLens.
2. Run the full model once over the draft sequence.
3. Compute `p/q` acceptance probabilities.
4. Accept a prefix of the draft.
5. Sample one correction token from the full model distribution if needed.

In [ ]:
stats = DecodeStats()

updated_ids, stats = speculative_decode_once(
    model=model,
    tuned_lens=tuned_lens,
    all_ids=input_ids,
    layer_index=config.layer_index,
    gamma=config.gamma,
    stats=stats,
)

print("old length:", len(input_ids))
print("new length:", len(updated_ids))
print("accepted:", stats.accepted)
print("generated:", stats.generated)
print("alpha:", stats.alpha)
print("new text:")
print(tokenizer.decode(updated_ids))

## 10. Run full speculative generation for one example

This repeatedly applies the speculative step until `seqlen` new tokens have been generated.

In [ ]:
spec_out = speculative_generate(
    model=model,
    tuned_lens=tuned_lens,
    input_ids=input_ids,
    seqlen=config.seqlen,
    layer_index=config.layer_index,
    gamma=config.gamma,
)

spec_ids = spec_out["sequences"][0]
spec_stats = spec_out["stats"]

print("generated new tokens:", len(spec_ids) - len(input_ids))
print("accepted:", spec_stats.accepted)
print("generated:", spec_stats.generated)
print("alpha:", f"{spec_stats.alpha:.2%}")
print()
print(tokenizer.decode(spec_ids))

## 11. Benchmark one example against vanilla generation

This compares:

```text
early-exit tuned-lens speculative generation
vs.
regular model.generate()
```

A negative improvement means the speculative pipeline is slower.

That is common here because this is a research/prototype version, not a kernel-optimized serving implementation.

In [ ]:
one_result = benchmark_one_example(
    model=model,
    tokenizer=tokenizer,
    tuned_lens=tuned_lens,
    text=input_text,
    config=config,
)

one_result

## 12. Run the full benchmark loop

This loads the dataset stream and runs `num_examples` examples.

Start with `num_examples=10`. Increase to `100` after the pipeline works.

In [ ]:
results = run_benchmark(config)

print("num results:", len(results))
print("first result:")
print(json.dumps(results[0], indent=2) if results else "no results")

## 13. Summarize results

Key metrics:

- `alpha`: accepted draft tokens / generated tokens
- `speculative_time`: time for early-exit speculative decoding
- `vanilla_time`: time for normal generation
- `improvement`: `1 - speculative_time / vanilla_time`

Interpretation:

```text
improvement > 0  => speculative was faster
improvement = 0  => same speed
improvement < 0  => speculative was slower
```

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

if len(df) > 0:
    summary = {
        "num_examples": len(df),
        "mean_alpha": df["alpha"].mean(),
        "mean_speculative_time": df["speculative_time"].mean(),
        "mean_vanilla_time": df["vanilla_time"].mean(),
        "mean_improvement": df["improvement"].mean(),
        "total_accepted": df["accepted"].sum(),
        "total_generated": df["generated"].sum(),
        "global_alpha": df["accepted"].sum() / max(df["generated"].sum(), 1),
    }

    print(json.dumps(summary, indent=2))
    display(df.head())
else:
    print("No benchmark results.")

## 14. Save results to JSON

This saves per-example results and a summary.

In [ ]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

results_path = results_dir / "early_exit_benchmark_results.json"
summary_path = results_dir / "early_exit_benchmark_summary.json"

if len(results) > 0:
    results_path.write_text(json.dumps(results, indent=2))

    summary = {
        "num_examples": len(df),
        "mean_alpha": float(df["alpha"].mean()),
        "mean_speculative_time": float(df["speculative_time"].mean()),
        "mean_vanilla_time": float(df["vanilla_time"].mean()),
        "mean_improvement": float(df["improvement"].mean()),
        "total_accepted": float(df["accepted"].sum()),
        "total_generated": float(df["generated"].sum()),
        "global_alpha": float(df["accepted"].sum() / max(df["generated"].sum(), 1)),
        "config": {
            "model_name": config.model_name,
            "gamma": config.gamma,
            "seqlen": config.seqlen,
            "layer_index": config.layer_index,
            "num_prefix_tokens": config.num_prefix_tokens,
            "num_examples": config.num_examples,
            "device": config.device,
            "dtype": str(config.dtype),
        },
    }
    summary_path.write_text(json.dumps(summary, indent=2))

print("wrote:", results_path)
print("wrote:", summary_path)

## 15. Optional: run the same pipeline through CLI

This uses the modular `.py` entry point:

```bash
python -m early_exit.run_benchmark ...
```

In [ ]:
# Optional CLI run. Uncomment to use.
# !python -m early_exit.run_benchmark \
#   --model-name EleutherAI/pythia-1.4b-deduped \
#   --device cuda \
#   --dtype bfloat16 \
#   --gamma 4 \
#   --seqlen 10 \
#   --layer-index 3 \
#   --num-prefix-tokens 10 \
#   --num-examples 10 \
#   --out results/early_exit_cli_results.json

## 16. Suggested experiments

Try a grid over layer and gamma:

```text
layer_index in [2, 4, 6, 8, 10, 12]
gamma in [2, 4, 6]
```

Expected tradeoff:

```text
early layer  -> cheaper draft, lower alpha
deeper layer -> more expensive draft, higher alpha
higher gamma -> more potential speedup, but more wasted draft work if alpha is low
```

For this prototype, negative improvement is not surprising. The goal is to study acceptance behavior and early-layer quality. Real speedups need a cheaper draft path and less Python/model-mutation overhead.